In [1]:
# 学習
import torch
import torch.optim as optim
from yolov3.models.yolo import Model
import yaml
from tqdm import tqdm
from src.domain.loss import CustomLoss
from src.domain.dataloader import CustomDataset, custom_collate_fn
from torch.utils.data import DataLoader

# モデルのロード
config_path = 'yolov3/models/yolov5s.yaml'
model_path = 'models/pre_trained/yolov5s.pt'
model = Model(config_path)
model.load_state_dict(torch.load(model_path)['model'].state_dict())  # yolov5s.ptは、学習済みの重みファイル
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

hyp_path = "data/hyps/hyp.scratch-low.yaml"
with open(hyp_path, errors="ignore") as f:
    hyp = yaml.safe_load(f)

model.hyp = hyp

# カスタム損失関数
criterion = CustomLoss(model)

# オプティマイザ
optimizer = optim.Adam(model.parameters(), lr=0.001)
# scaler = torch.cuda.amp.GradScaler(enabled=True) # 高速化ライブラリ必要であれば利用したい

# データローダ
img_dir = './data/coco128/images/train2017'
annotation_dir = './data/coco128/labels/train2017'
train_dataset = CustomDataset(
    img_dir=img_dir,
    annotation_dir=annotation_dir,
)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=custom_collate_fn)

# tqdmの表示フォーマット
TQDM_BAR_FORMAT = '{l_bar}{bar:10}{r_bar}'

# トレーニングループ
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, total=len(train_loader), bar_format=TQDM_BAR_FORMAT)
    for images, targets in pbar:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss, loss_items = criterion(outputs, targets)
        # scaler.scale(loss).backward()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        pbar.set_description(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / len(train_loader)}')
    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / len(train_loader)}')
    

# トレーニング済みモデルの保存
torch.save(model.state_dict(), 'models/fine_tuned/yolov5s_finetuned.pth')
print("model save!")


/home/docker/.cache/pypoetry/virtualenvs/repo-jK08sqZf-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: attempted relative import with no known parent package